<a href="https://colab.research.google.com/github/Shiva-Kumar-S-M/Placement-Training/blob/main/Hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install transformers datasets accelerate evaluate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
from datasets import Dataset
from transformers import DistilBertTokenizerFast,DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import evaluate
import numpy as np
from sklearn.model_selection import train_test_split



In [4]:
df=pd.read_csv("/content/IMDB Dataset.csv.zip")

In [6]:
df['label']=df['sentiment'].map({'positive':1,'negative':0})
df=df[['review','label']]
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['label'])
train_ds=Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df.reset_index(drop=True))
train_ds,test_ds


(Dataset({
     features: ['review', 'label'],
     num_rows: 40000
 }),
 Dataset({
     features: ['review', 'label'],
     num_rows: 10000
 }))

In [7]:
tokenizer=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
  return tokenizer(batch['review'],padding=True,truncation=True,max_length=256)

train_ds=train_ds.map(tokenize,batched=True,batch_size=None)
test_ds=test_ds.map(tokenize,batched=True,batch_size=None)

print("\n Columns after tokenization")
print(train_ds.column_names)

print("first tokenized example ")
print(train_ds[0])

cols=['input_ids','attention_mask','label']
train_ds.set_format(type='torch',columns=cols)
test_ds.set_format(type='torch',columns=cols)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]


 Columns after tokenization
['review', 'label', 'input_ids', 'attention_mask']
first tokenized example 
{'review': 'I caught this little gem totally by accident back in 1980 or \'81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, within seconds, the audience was in hysterics! The biggest laugh came when they showed "Princess Laia" having huge cinnamon buns instead of hair on her head. She looks at the camera, gives a grim smile and nods. That made it even funnier! You gotta see "Chewabacca" played by what looks like a Muppet! It was extremely silly and stupid...but I couldn\'t stop laughing. Most of the dialogue was drowned out because of all the laughter. Also if you know "Star Wars" pretty well it\'s even funnier--they deliberately poke fun at some of the dialogue. This REALLY works with an audience! A 

In [8]:
model=DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
accuracy=evaluate.load("accuracy")
precision=evaluate.load('precision')
recall=evaluate.load('recall')
f1=evaluate.load('f1')


def compute_metrics(eval_pred):
  logits,labels=eval_pred
  predictions=np.argmax(logits,axis=-1)
  return {'accuracy':accuracy.compute(predictions=predictions,references=labels)['accuracy'],
          'precision':precision.compute(predictions=predictions,references=labels)['precision'],
          'recall':recall.compute(predictions=predictions,references=labels)['recall'],
          'f1':f1.compute(predictions=predictions,references=labels)['f1'] }

In [10]:
training_args=TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100 )



In [ ]:
traine